# L09 — Python Tools for Simulation

**Module**: M04 | **Chapter**: 6 | **Lecture**: L09

## Learning Objectives
By the end of this notebook you will be able to:
1. Generate reproducible random variates using `numpy.random.default_rng`.
2. Fit and evaluate probability distributions with `scipy.stats`.
3. Collect simulation output into a `pandas` DataFrame and summarize it.
4. Plot histograms and empirical CDFs with `matplotlib`.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
# Standard imports — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

print(f"NumPy  {np.__version__}")
print(f"pandas {pd.__version__}")

## 1. Reproducible Random Numbers

Use `numpy.random.default_rng(seed)` — not `np.random.seed()`.  
The new interface is statistically superior and produces independent streams.

In [ ]:
rng = np.random.default_rng(seed=42)

# Exponential interarrival times (mean = 1/lambda = 0.25 min)
lam = 4.0   # customers per minute
interarrivals = rng.exponential(scale=1.0 / lam, size=10_000)

print(f"Sample mean : {interarrivals.mean():.4f}  (theory: {1/lam:.4f})")
print(f"Sample std  : {interarrivals.std():.4f}  (theory: {1/lam:.4f})")

In [ ]:
# Spawn independent child generators for replications
base_rng = np.random.default_rng(seed=0)
child_rngs = base_rng.spawn(5)

for i, child in enumerate(child_rngs):
    sample = child.exponential(1.0, size=1_000)
    print(f"Rep {i}: mean = {sample.mean():.4f}")

## 2. Common Distributions with SciPy

In [ ]:
# Exponential distribution object
dist = stats.expon(scale=0.25)   # mean = 0.25

print(f"Mean  : {dist.mean():.4f}")
print(f"Var   : {dist.var():.4f}")
print(f"P(X<=0.5): {dist.cdf(0.5):.4f}")
print(f"95th pct  : {dist.ppf(0.95):.4f}")

In [ ]:
# Plot histogram vs PDF
rng2 = np.random.default_rng(7)
samples = rng2.exponential(0.25, size=5_000)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(samples, bins=50, density=True, alpha=0.6,
        color='steelblue', label='Histogram (n=5000)')

x = np.linspace(0, 1.5, 300)
ax.plot(x, dist.pdf(x), 'r-', lw=2, label='Exponential PDF')

ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.set_title('Exponential(mean=0.25) — histogram vs PDF')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Collecting Output with pandas

In [ ]:
# Simulate a simple M/M/1 queue manually (pedagogical, not production code)
# We'll use the simdes package in L10; here we build from scratch.

def simulate_mm1(lam: float, mu: float, n_customers: int, seed: int) -> pd.DataFrame:
    """Simulate M/M/1 queue; return per-customer DataFrame."""
    rng = np.random.default_rng(seed)
    records = []
    t = 0.0          # current time
    server_free_at = 0.0  # when server becomes free next

    for i in range(n_customers):
        t += rng.exponential(1.0 / lam)   # arrival time
        start = max(t, server_free_at)    # service start
        wait = start - t
        svc = rng.exponential(1.0 / mu)
        server_free_at = start + svc
        records.append({'id': i, 'arrival': t, 'wait': wait,
                        'service': svc, 'sojourn': wait + svc})
    return pd.DataFrame(records)

df = simulate_mm1(lam=3.0, mu=4.0, n_customers=10_000, seed=42)
df[['wait', 'sojourn']].describe().round(4)

In [ ]:
# Compare to M/M/1 theory
lam, mu = 3.0, 4.0
rho = lam / mu
Wq_theory = lam / (mu * (mu - lam))
W_theory = 1.0 / (mu - lam)

print(f"Simulated  Wq = {df['wait'].mean():.4f}  (theory: {Wq_theory:.4f})")
print(f"Simulated  W  = {df['sojourn'].mean():.4f}  (theory: {W_theory:.4f})")

## 4. Multi-Replication Summary

In [ ]:
results = []
for rep in range(30):
    df_rep = simulate_mm1(lam=3.0, mu=4.0, n_customers=5_000, seed=rep)
    results.append({'rep': rep,
                    'mean_wait': df_rep['wait'].mean(),
                    'mean_sojourn': df_rep['sojourn'].mean()})

summary = pd.DataFrame(results)
print(summary[['mean_wait', 'mean_sojourn']].describe().round(4))

# 95% CI for mean_wait
from scipy.stats import t as t_dist
n = len(summary)
mean = summary['mean_wait'].mean()
se = summary['mean_wait'].std() / np.sqrt(n)
half_width = t_dist.ppf(0.975, df=n-1) * se
print(f"\n95% CI for Wq: ({mean-half_width:.4f}, {mean+half_width:.4f})")
print(f"Theory Wq    : {Wq_theory:.4f}")

In [ ]:
# Box plot of mean wait across replications
fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot(summary['mean_wait'], vert=True, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.axhline(Wq_theory, color='red', linestyle='--', label=f'Theory Wq={Wq_theory:.3f}')
ax.set_ylabel('Mean wait time (min)')
ax.set_title('M/M/1 Wq — 30 replications (n=5000 each)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Try It Yourself

1. Change `lam=3.8` (very high load, ρ=0.95). How wide is the 95% CI? How many customers per replication would you need to halve the half-width?

2. Replace the Exponential service with `rng.lognormal(mean=np.log(1/mu), sigma=0.5)` and repeat. Do the simulated means still match M/M/1 theory? Why or why not?

3. Plot the empirical CDF of wait times using `plt.ecdf` (matplotlib ≥ 3.8) or `np.sort` + cumulative sum. Overlay the theoretical Exponential CDF. Are they a good match?